In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. DATA LOADING
# ============================================================================

# Load your data (adjust file paths as needed)
fact_drilling = pd.read_csv(r"C:\Users\PCexpress\Downloads\fact_drilling_operations.csv")
dim_well = pd.read_csv(r"C:\Users\PCexpress\Downloads\dim_well.csv")
dim_rig = pd.read_csv(r"C:\Users\PCexpress\Downloads\dim_rig.csv")
dim_time = pd.read_csv(r"C:\Users\PCexpress\Downloads\dim_time.csv")
dim_drilling_params = pd.read_csv(r"C:\Users\PCexpress\Downloads\dim_drilling_parameters.csv")
dim_anomalies = pd.read_csv(r"C:\Users\PCexpress\Downloads\dim_anomalies.csv")

# ============================================================================
# 2. DATA CLEANING & PREPROCESSING
# ============================================================================

def clean_drilling_data(fact_drilling, dim_well, dim_rig, dim_time, dim_drilling_params, dim_anomalies):
    """
    Clean and prepare drilling operations data for analysis
    """
    
    # --- FACT TABLE CLEANING ---
    
    print("Missing values before cleaning:")
    print(fact_drilling.isnull().sum())
    
    # Remove duplicates
    fact_drilling = fact_drilling.drop_duplicates(subset=['operation_id'])
    
    # Handle missing numerical values
    numerical_cols = ['depth_m', 'rop_m_per_hr', 'wob_klbf', 'torque_klbf_ft', 
                      'rpm', 'mud_flow_rate_gpm', 'standpipe_pressure_psi',
                      'mechanical_specific_energy', 'drilling_cost_usd']
    
    for col in numerical_cols:
        if col in fact_drilling.columns:
            # Replace negative values with NaN (except cost which can be 0)
            if col != 'drilling_cost_usd':
                fact_drilling.loc[fact_drilling[col] < 0, col] = np.nan
            
            # Fill missing values with median by rig
            fact_drilling[col] = fact_drilling.groupby('rig_id')[col].transform(
                lambda x: x.fillna(x.median())
            )
            
            # If still missing, fill with overall median
            fact_drilling[col].fillna(fact_drilling[col].median(), inplace=True)
    
    # Clean depth - ensure it's positive and reasonable
    fact_drilling['depth_m'] = fact_drilling['depth_m'].clip(lower=0, upper=15000)
    
    # Clean ROP - remove unrealistic values
    fact_drilling['rop_m_per_hr'] = fact_drilling['rop_m_per_hr'].clip(lower=0, upper=200)
    
    # Clean WOB - typical range
    fact_drilling['wob_klbf'] = fact_drilling['wob_klbf'].clip(lower=0, upper=100)
    
    # Clean RPM - typical range
    fact_drilling['rpm'] = fact_drilling['rpm'].clip(lower=0, upper=300)
    
    # Clean MSE - remove outliers
    mse_q99 = fact_drilling['mechanical_specific_energy'].quantile(0.99)
    fact_drilling['mechanical_specific_energy'] = fact_drilling['mechanical_specific_energy'].clip(
        lower=0, upper=mse_q99
    )
    
    # Clean NPT flag
    fact_drilling['is_npt_flag'] = fact_drilling['is_npt_flag'].map({
        1: 1, '1': 1, 'Yes': 1, 'yes': 1, 'Y': 1, True: 1,
        0: 0, '0': 0, 'No': 0, 'no': 0, 'N': 0, False: 0
    }).fillna(0).astype(int)
    
    # Clean drilling cost
    fact_drilling['drilling_cost_usd'] = fact_drilling['drilling_cost_usd'].clip(lower=0)
    fact_drilling['drilling_cost_usd'].fillna(0, inplace=True)
    
    # --- DIMENSION TABLES CLEANING ---
    
    # dim_well
    dim_well = dim_well.drop_duplicates(subset=['well_id'])
    dim_well['spud_date'] = pd.to_datetime(dim_well['spud_date'], errors='coerce')
    dim_well['target_depth_m'] = pd.to_numeric(dim_well['target_depth_m'], errors='coerce')
    dim_well['target_depth_m'].fillna(dim_well['target_depth_m'].median(), inplace=True)
    dim_well['well_type'].fillna('Unknown', inplace=True)
    
    # dim_rig
    dim_rig = dim_rig.drop_duplicates(subset=['rig_id'])
    dim_rig['daily_cost_usd'] = pd.to_numeric(dim_rig['daily_cost_usd'], errors='coerce')
    dim_rig['daily_cost_usd'].fillna(dim_rig['daily_cost_usd'].median(), inplace=True)
    dim_rig['max_depth_m'] = pd.to_numeric(dim_rig['max_depth_m'], errors='coerce')
    dim_rig['horsepower'] = pd.to_numeric(dim_rig['horsepower'], errors='coerce')
    dim_rig['rig_type'].fillna('Unknown', inplace=True)
    
    # dim_time
    dim_time = dim_time.drop_duplicates(subset=['time_id'])
    dim_time['date'] = pd.to_datetime(dim_time['date'], errors='coerce')
    
    # dim_anomalies
    dim_anomalies = dim_anomalies.drop_duplicates(subset=['anomaly_id'])
    dim_anomalies['severity'].fillna('Low', inplace=True)
    dim_anomalies['auto_shutdown_flag'] = dim_anomalies['auto_shutdown_flag'].map({
        'Yes': 1, 'yes': 1, 'Y': 1, 1: 1, True: 1,
        'No': 0, 'no': 0, 'N': 0, 0: 0, False: 0
    }).fillna(0).astype(int)
    
    print("\nMissing values after cleaning:")
    print(fact_drilling.isnull().sum())
    
    return fact_drilling, dim_well, dim_rig, dim_time, dim_drilling_params, dim_anomalies


# ============================================================================
# 3. FEATURE ENGINEERING
# ============================================================================

def engineer_features(fact_drilling, dim_rig, dim_time):
    """
    Create derived features for KPI calculation
    """
    
    # Merge rig daily cost
    fact_drilling = fact_drilling.merge(
        dim_rig[['rig_id', 'daily_cost_usd', 'rig_name', 'rig_type']], 
        on='rig_id', 
        how='left'
    )
    
    # Merge time information
    fact_drilling = fact_drilling.merge(
        dim_time[['time_id', 'date', 'hour', 'shift', 'day', 'month', 'year']], 
        on='time_id', 
        how='left'
    )
    
    # Calculate hourly cost (daily cost / 24 hours)
    fact_drilling['cost_per_hour'] = fact_drilling['daily_cost_usd'] / 24
    
    # Add drilling cost to hourly cost for total cost
    fact_drilling['total_cost_per_hour'] = (
        fact_drilling['cost_per_hour'] + fact_drilling['drilling_cost_usd']
    )
    
    # Calculate cost per meter
    fact_drilling['cost_per_meter'] = np.where(
        fact_drilling['rop_m_per_hr'] > 0,
        fact_drilling['total_cost_per_hour'] / fact_drilling['rop_m_per_hr'],
        np.nan
    )
    
    # Fill infinite or very high cost per meter values
    cost_per_meter_q95 = fact_drilling['cost_per_meter'].quantile(0.95)
    fact_drilling['cost_per_meter'] = fact_drilling['cost_per_meter'].clip(upper=cost_per_meter_q95)
    
    # Calculate drilling efficiency index (ROP / MSE ratio normalized)
    fact_drilling['drilling_efficiency_index'] = np.where(
        fact_drilling['mechanical_specific_energy'] > 0,
        fact_drilling['rop_m_per_hr'] / fact_drilling['mechanical_specific_energy'],
        0
    )
    
    # Normalize drilling efficiency (0-100 scale)
    max_efficiency = fact_drilling['drilling_efficiency_index'].quantile(0.95)
    fact_drilling['drilling_efficiency_index'] = (
        (fact_drilling['drilling_efficiency_index'] / max_efficiency) * 100
    ).clip(upper=100)
    
    # Calculate NPT hours (1 hour per record if NPT flag is set)
    fact_drilling['npt_hours'] = fact_drilling['is_npt_flag']
    
    # Calculate productive hours
    fact_drilling['productive_hours'] = 1 - fact_drilling['is_npt_flag']
    
    # Cost during NPT
    fact_drilling['cost_during_npt'] = (
        fact_drilling['total_cost_per_hour'] * fact_drilling['is_npt_flag']
    )
    
    return fact_drilling


# ============================================================================
# 4. MERGE ALL DIMENSIONS
# ============================================================================

def merge_all_dimensions(fact_drilling, dim_well, dim_rig, dim_time, dim_drilling_params):
    """
    Create complete dataset with all dimensions
    """
    
    df = fact_drilling.copy()
    
    # Merge well information
    df = df.merge(
        dim_well[['well_id', 'well_name', 'field_name', 'basin', 'country', 
                  'well_type', 'target_depth_m', 'operator']], 
        on='well_id', 
        how='left'
    )
    
    # Calculate depth progress percentage
    df['depth_progress_pct'] = (df['depth_m'] / df['target_depth_m'] * 100).clip(upper=100)
    
    # Calculate remaining depth
    df['remaining_depth_m'] = (df['target_depth_m'] - df['depth_m']).clip(lower=0)
    
    return df


# ============================================================================
# 5. PERFORMANCE KPIs
# ============================================================================

def calculate_performance_kpis(df):
    """
    Calculate drilling performance KPIs
    """
    
    kpis = {}
    
    # 1. Average ROP (m/hr)
    kpis['avg_rop_m_per_hr'] = df['rop_m_per_hr'].mean()
    
    # 2. Cost per Meter ($/m)
    kpis['avg_cost_per_meter'] = df['cost_per_meter'].mean()
    
    # 3. NPT %
    total_hours = len(df)
    npt_hours = df['is_npt_flag'].sum()
    kpis['npt_percentage'] = (npt_hours / total_hours * 100) if total_hours > 0 else 0
    
    # 4. Drilling Efficiency Index (0-100)
    kpis['avg_drilling_efficiency_index'] = df['drilling_efficiency_index'].mean()
    
    # 5. Rig Utilization %
    productive_hours = df['productive_hours'].sum()
    kpis['rig_utilization_pct'] = (productive_hours / total_hours * 100) if total_hours > 0 else 0
    
    # 6. Total Drilling Cost
    kpis['total_drilling_cost_usd'] = df['total_cost_per_hour'].sum()
    
    # 7. Average Cost per Hour
    kpis['avg_cost_per_hour'] = df['total_cost_per_hour'].mean()
    
    # 8. Total NPT Hours
    kpis['total_npt_hours'] = npt_hours
    
    # 9. Total Cost During NPT
    kpis['total_cost_during_npt'] = df['cost_during_npt'].sum()
    
    # 10. Average Depth Progress
    kpis['avg_depth_progress_pct'] = df['depth_progress_pct'].mean()
    
    # 11. MSE Deviation % (from optimal - assuming optimal is 25th percentile)
    optimal_mse = df['mechanical_specific_energy'].quantile(0.25)
    actual_mse = df['mechanical_specific_energy'].mean()
    kpis['mse_deviation_pct'] = ((actual_mse - optimal_mse) / optimal_mse * 100) if optimal_mse > 0 else 0
    
    return pd.Series(kpis)


# ============================================================================
# 6. ROP ANALYSIS BY WELL AND RIG
# ============================================================================

def calculate_rop_by_well_rig(df):
    """
    ROP analysis by well and rig
    """
    
    # ROP by Well
    rop_by_well = df.groupby(['well_id', 'well_name']).agg({
        'rop_m_per_hr': 'mean',
        'depth_m': 'max',
        'drilling_efficiency_index': 'mean',
        'cost_per_meter': 'mean',
        'operation_id': 'count'
    }).reset_index()
    
    rop_by_well.columns = [
        'well_id', 'well_name', 'avg_rop', 'max_depth', 
        'avg_efficiency', 'avg_cost_per_meter', 'total_operations'
    ]
    
    # ROP by Rig
    rop_by_rig = df.groupby(['rig_id', 'rig_name']).agg({
        'rop_m_per_hr': 'mean',
        'depth_m': 'max',
        'drilling_efficiency_index': 'mean',
        'cost_per_meter': 'mean',
        'is_npt_flag': 'mean',
        'operation_id': 'count'
    }).reset_index()
    
    rop_by_rig.columns = [
        'rig_id', 'rig_name', 'avg_rop', 'max_depth', 
        'avg_efficiency', 'avg_cost_per_meter', 'npt_rate', 'total_operations'
    ]
    rop_by_rig['npt_rate'] = rop_by_rig['npt_rate'] * 100
    
    return rop_by_well, rop_by_rig


# ============================================================================
# 7. ROP TREND VS DEPTH
# ============================================================================

def calculate_rop_vs_depth(df):
    """
    Analyze ROP trends with depth intervals
    """
    
    # Create depth bins (every 500m)
    df['depth_bin'] = (df['depth_m'] // 500) * 500
    
    rop_depth_trend = df.groupby('depth_bin').agg({
        'rop_m_per_hr': ['mean', 'std', 'count'],
        'mechanical_specific_energy': 'mean',
        'drilling_efficiency_index': 'mean',
        'wob_klbf': 'mean',
        'rpm': 'mean'
    }).reset_index()
    
    rop_depth_trend.columns = [
        'depth_bin_m', 'avg_rop', 'rop_std', 'sample_count',
        'avg_mse', 'avg_efficiency', 'avg_wob', 'avg_rpm'
    ]
    
    return rop_depth_trend


# ============================================================================
# 8. RPM & WOB STABILITY
# ============================================================================

def calculate_stability_metrics(df):
    """
    Calculate RPM and WOB stability metrics
    """
    
    # Calculate coefficient of variation (CV) for each well/rig
    stability_by_well = df.groupby(['well_id', 'well_name']).agg({
        'rpm': ['mean', 'std'],
        'wob_klbf': ['mean', 'std'],
        'torque_klbf_ft': ['mean', 'std'],
        'operation_id': 'count'
    }).reset_index()
    
    stability_by_well.columns = [
        'well_id', 'well_name', 
        'rpm_mean', 'rpm_std', 
        'wob_mean', 'wob_std',
        'torque_mean', 'torque_std',
        'sample_count'
    ]
    
    # Calculate coefficient of variation (lower is better - more stable)
    stability_by_well['rpm_cv'] = (
        stability_by_well['rpm_std'] / stability_by_well['rpm_mean'] * 100
    )
    stability_by_well['wob_cv'] = (
        stability_by_well['wob_std'] / stability_by_well['wob_mean'] * 100
    )
    
    # Overall stability score (inverse of CV, normalized)
    stability_by_well['rpm_stability_score'] = (
        100 - stability_by_well['rpm_cv'].clip(upper=100)
    )
    stability_by_well['wob_stability_score'] = (
        100 - stability_by_well['wob_cv'].clip(upper=100)
    )
    
    return stability_by_well


# ============================================================================
# 9. COST ANALYSIS
# ============================================================================

def calculate_cost_kpis(df):
    """
    Detailed cost analysis
    """
    
    # Cost by Rig
    cost_by_rig = df.groupby(['rig_id', 'rig_name']).agg({
        'total_cost_per_hour': 'sum',
        'cost_during_npt': 'sum',
        'cost_per_meter': 'mean',
        'drilling_cost_usd': 'sum',
        'operation_id': 'count',
        'depth_m': 'max'
    }).reset_index()
    
    cost_by_rig.columns = [
        'rig_id', 'rig_name', 'total_cost', 'npt_cost', 
        'avg_cost_per_meter', 'total_drilling_cost', 'hours_operated', 'max_depth'
    ]
    
    # Cost by Well
    cost_by_well = df.groupby(['well_id', 'well_name']).agg({
        'total_cost_per_hour': 'sum',
        'cost_during_npt': 'sum',
        'cost_per_meter': 'mean',
        'drilling_cost_usd': 'sum',
        'operation_id': 'count',
        'depth_m': 'max'
    }).reset_index()
    
    cost_by_well.columns = [
        'well_id', 'well_name', 'total_cost', 'npt_cost', 
        'avg_cost_per_meter', 'total_drilling_cost', 'hours_operated', 'max_depth'
    ]
    
    # Calculate NPT cost percentage
    cost_by_rig['npt_cost_pct'] = (
        cost_by_rig['npt_cost'] / cost_by_rig['total_cost'] * 100
    )
    cost_by_well['npt_cost_pct'] = (
        cost_by_well['npt_cost'] / cost_by_well['total_cost'] * 100
    )
    
    return cost_by_rig, cost_by_well


# ============================================================================
# 10. SAFETY & ANOMALY KPIs
# ============================================================================

def calculate_safety_kpis(df, dim_anomalies):
    """
    Calculate safety-related KPIs
    Note: This assumes anomaly data is linked to operations via operation_id
    You may need to adjust based on your actual data structure
    """
    
    # Create sample anomaly-operation link if not exists
    # In real scenario, you'd have a fact table linking operations to anomalies
    
    kpis = {}
    
    # If you have anomaly_id in fact table, use this approach:
    # Otherwise, create synthetic data for demonstration
    
    # Anomaly frequency
    kpis['total_anomalies'] = len(dim_anomalies)
    kpis['anomaly_frequency_per_1000_ops'] = (
        len(dim_anomalies) / len(df) * 1000
    ) if len(df) > 0 else 0
    
    # High severity events
    high_severity = dim_anomalies[dim_anomalies['severity'].isin(['High', 'high', 'HIGH'])]
    kpis['high_severity_events'] = len(high_severity)
    
    # Auto shutdown events
    kpis['auto_shutdown_events'] = dim_anomalies['auto_shutdown_flag'].sum()
    kpis['auto_shutdown_pct'] = (
        kpis['auto_shutdown_events'] / kpis['total_anomalies'] * 100
    ) if kpis['total_anomalies'] > 0 else 0
    
    # Safety critical events (assuming High severity + Safety risk category)
    safety_critical = dim_anomalies[
        (dim_anomalies['severity'].isin(['High', 'high', 'HIGH'])) &
        (dim_anomalies['risk_category'].str.contains('Safety', case=False, na=False))
    ]
    kpis['safety_critical_events'] = len(safety_critical)
    
    # Kick/Overpressure events
    kick_events = dim_anomalies[
        dim_anomalies['anomaly_type'].str.contains('Kick', case=False, na=False)
    ]
    kpis['kick_overpressure_events'] = len(kick_events)
    
    # Calculate Safety Risk Index (weighted score)
    # High severity = 3 points, Medium = 2, Low = 1
    severity_weights = {'High': 3, 'high': 3, 'HIGH': 3, 
                       'Medium': 2, 'medium': 2, 'MEDIUM': 2,
                       'Low': 1, 'low': 1, 'LOW': 1}
    
    dim_anomalies['severity_score'] = dim_anomalies['severity'].map(severity_weights).fillna(1)
    kpis['safety_risk_index'] = dim_anomalies['severity_score'].sum()
    
    # Anomaly breakdown by type
    anomaly_by_type = dim_anomalies.groupby('anomaly_type').agg({
        'anomaly_id': 'count',
        'severity_score': 'sum',
        'auto_shutdown_flag': 'sum'
    }).reset_index()
    
    anomaly_by_type.columns = [
        'anomaly_type', 'count', 'total_severity_score', 'auto_shutdowns'
    ]
    anomaly_by_type = anomaly_by_type.sort_values('count', ascending=False)
    
    return pd.Series(kpis), anomaly_by_type


# ============================================================================
# 11. DEPTH PROGRESS ANALYSIS
# ============================================================================

def calculate_depth_progress(df):
    """
    Analyze depth progress vs target
    """
    
    progress_by_well = df.groupby(['well_id', 'well_name']).agg({
        'depth_m': 'max',
        'target_depth_m': 'first',
        'depth_progress_pct': 'max',
        'remaining_depth_m': 'min',
        'rop_m_per_hr': 'mean'
    }).reset_index()
    
    progress_by_well.columns = [
        'well_id', 'well_name', 'current_depth', 'target_depth',
        'progress_pct', 'remaining_depth', 'avg_rop'
    ]
    
    # Estimate remaining time (hours)
    progress_by_well['estimated_remaining_hours'] = (
        progress_by_well['remaining_depth'] / progress_by_well['avg_rop']
    )
    
    # Status classification
    progress_by_well['status'] = progress_by_well['progress_pct'].apply(
        lambda x: 'Completed' if x >= 100 else 
                  'Near Completion' if x >= 80 else
                  'In Progress' if x >= 50 else
                  'Early Stage'
    )
    
    return progress_by_well


# ============================================================================
# 12. MONTHLY TRENDS
# ============================================================================

def calculate_monthly_trends(df):
    """
    Calculate KPI trends over time
    """
    
    monthly = df.groupby(['year', 'month']).agg({
        'rop_m_per_hr': 'mean',
        'cost_per_meter': 'mean',
        'drilling_efficiency_index': 'mean',
        'is_npt_flag': 'mean',
        'total_cost_per_hour': 'sum',
        'mechanical_specific_energy': 'mean',
        'depth_m': 'max',
        'operation_id': 'count'
    }).reset_index()
    
    monthly.columns = [
        'year', 'month', 'avg_rop', 'avg_cost_per_meter', 
        'avg_efficiency', 'npt_rate', 'total_cost', 
        'avg_mse', 'max_depth', 'operations_count'
    ]
    
    monthly['npt_rate'] = monthly['npt_rate'] * 100
    
    return monthly


# ============================================================================
# 13. MAIN EXECUTION
# ============================================================================

# Clean data
fact_drilling, dim_well, dim_rig, dim_time, dim_drilling_params, dim_anomalies = clean_drilling_data(
    fact_drilling, dim_well, dim_rig, dim_time, dim_drilling_params, dim_anomalies
)

# Engineer features
fact_drilling = engineer_features(fact_drilling, dim_rig, dim_time)

# Merge all dimensions
df_merged = merge_all_dimensions(fact_drilling, dim_well, dim_rig, dim_time, dim_drilling_params)

# ============================================================================
# CALCULATE ALL KPIs
# ============================================================================

print("\n" + "="*80)
print("OVERALL PERFORMANCE KPIs")
print("="*80)
performance_kpis = calculate_performance_kpis(df_merged)
print(performance_kpis)

print("\n" + "="*80)
print("ROP BY WELL (Top 10)")
print("="*80)
rop_by_well, rop_by_rig = calculate_rop_by_well_rig(df_merged)
print(rop_by_well.sort_values('avg_rop', ascending=False).head(10))

print("\n" + "="*80)
print("ROP BY RIG")
print("="*80)
print(rop_by_rig.sort_values('avg_rop', ascending=False))

print("\n" + "="*80)
print("ROP TREND VS DEPTH")
print("="*80)
rop_depth_trend = calculate_rop_vs_depth(df_merged)
print(rop_depth_trend.head(20))

print("\n" + "="*80)
print("RPM & WOB STABILITY (Top 10 Most Stable Wells)")
print("="*80)
stability_metrics = calculate_stability_metrics(df_merged)
print(stability_metrics.sort_values('rpm_stability_score', ascending=False).head(10))

print("\n" + "="*80)
print("COST ANALYSIS - BY RIG")
print("="*80)
cost_by_rig, cost_by_well = calculate_cost_kpis(df_merged)
print(cost_by_rig.sort_values('total_cost', ascending=False))

print("\n" + "="*80)
print("COST ANALYSIS - BY WELL (Top 10)")
print("="*80)
print(cost_by_well.sort_values('total_cost', ascending=False).head(10))

print("\n" + "="*80)
print("SAFETY & ANOMALY KPIs")
print("="*80)
safety_kpis, anomaly_by_type = calculate_safety_kpis(df_merged, dim_anomalies)
print(safety_kpis)
print("\nAnomaly Breakdown by Type:")
print(anomaly_by_type)

print("\n" + "="*80)
print("DEPTH PROGRESS VS TARGET")
print("="*80)
depth_progress = calculate_depth_progress(df_merged)
print(depth_progress.sort_values('progress_pct', ascending=False).head(10))

print("\n" + "="*80)
print("MONTHLY TRENDS (Last 12 Months)")
print("="*80)
monthly_trends = calculate_monthly_trends(df_merged)
print(monthly_trends.tail(12))


# ============================================================================
# 15. SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*80)
print("DATA SUMMARY")
print("="*80)
print(f"Total Operations: {len(df_merged):,}")
print(f"Total Wells: {df_merged['well_id'].nunique()}")
print(f"Total Rigs: {df_merged['rig_id'].nunique()}")
print(f"Date Range: {df_merged['date'].min()} to {df_merged['date'].max()}")
print(f"Depth Range: {df_merged['depth_m'].min():.0f}m to {df_merged['depth_m'].max():.0f}m")
print(f"Total Drilling Cost: ${df_merged['total_cost_per_hour'].sum():,.2f}")
print("="*80)

Missing values before cleaning:
operation_id                  0
time_id                       0
well_id                       0
rig_id                        0
depth_m                       0
rop_m_per_hr                  0
wob_klbf                      0
torque_klbf_ft                0
rpm                           0
mud_flow_rate_gpm             0
standpipe_pressure_psi        0
mechanical_specific_energy    0
drilling_cost_usd             0
is_npt_flag                   0
dtype: int64

Missing values after cleaning:
operation_id                  0
time_id                       0
well_id                       0
rig_id                        0
depth_m                       0
rop_m_per_hr                  0
wob_klbf                      0
torque_klbf_ft                0
rpm                           0
mud_flow_rate_gpm             0
standpipe_pressure_psi        0
mechanical_specific_energy    0
drilling_cost_usd             0
is_npt_flag                   0
dtype: int64

OVERALL PERFO